### Setup AWS Credentials

In [1]:
import os
# Setup your AWS Access Key and Secret Key as environment variables.
#os.environ["AWS_ACCESS_KEY_ID"]
#os.environ["AWS_SECRET_ACCESS_KEY"] 

import sys
sys.path.insert(0, '../../../src')

In [2]:
# Setup Nova Model (For SageMaker Inference this is just a label since the endpoint is the model)
NOVA_MODEL_ID = "nova-micro"

### Dataset Adapter

Initialize the Dataset Adapter that takes the input_columns and output_columns. We use the JSONDatasetAdapter to read a `.jsonl` file and adapt it to the standardized format. We also use the adapter to create train and test sets for our use case.

In [38]:
from amzn_nova_prompt_optimizer.core.input_adapters.dataset_adapter import JSONDatasetAdapter

input_columns = {"input"}
output_columns = {"answer"}

dataset_adapter = JSONDatasetAdapter(input_columns, output_columns)

# Adapt
dataset_adapter.adapt("../data/FacilitySupportAnalyzer.jsonl")

train_set, test_set = dataset_adapter.split(0.5)

### Prompt Adapter

Initialize the Prompt Adapter for the Original Prompt. For this example, we use the FacilitySupportAnalyzer User Prompt in the `.txt` format. 

In [4]:
from amzn_nova_prompt_optimizer.core.input_adapters.prompt_adapter import TextPromptAdapter

prompt_variables = input_columns

prompt_adapter = TextPromptAdapter()

prompt_adapter.set_user_prompt(file_path="original_prompt/user_prompt_as_template.txt", variables=prompt_variables)

# Adapt
prompt_adapter.adapt()

2026/01/29 19:33:26 INFO amzn_nova_prompt_optimizer.core.input_adapters.prompt_adapter: System Prompt not set, initializing as empty string...


### Metric Adapter

Initialize the Metric Adapter for evaluating this prompt for certain optimizers. For this example, we build a Custom Metric for the FacilitySupportAnalyzer Dataset. The metric adapter requires the use of the `apply` [For single row evaluation] or `batch_apply` [For evaluating the whole dataset together] function

In [5]:
from amzn_nova_prompt_optimizer.core.input_adapters.metric_adapter import MetricAdapter
from typing import List, Any, Dict
import re
import json

class FacilitySupportAnalyzerMetric(MetricAdapter):
    def parse_json(self, input_string: str):
        """
        Attempts to parse the given string as JSON. If direct parsing fails,
        it tries to extract a JSON snippet from code blocks formatted as:
            ```json
            ... JSON content ...
            ```
        or any code block delimited by triple backticks and then parses that content.
        """
        try:
            return json.loads(input_string)
        except json.JSONDecodeError as err:
            error = err

        patterns = [
            re.compile(r"```json\s*(.*?)\s*```", re.DOTALL | re.IGNORECASE),
            re.compile(r"```(.*?)```", re.DOTALL)
        ]

        for pattern in patterns:
            match = pattern.search(input_string)
            if match:
                json_candidate = match.group(1).strip()
                try:
                    return json.loads(json_candidate)
                except json.JSONDecodeError:
                    continue

        raise error

    def _calculate_metrics(self, y_pred: Any, y_true: Any) -> Dict:
        strict_json = False
        result = {
            "is_valid_json": False,
            "correct_categories": 0.0,
            "correct_sentiment": False,
            "correct_urgency": False,
        }

        try:
            y_true = y_true if isinstance(y_true, dict) else (json.loads(y_true) if strict_json else self.parse_json(y_true))
            y_pred = y_pred if isinstance(y_pred, dict) else (json.loads(y_pred) if strict_json else self.parse_json(y_pred))
        except json.JSONDecodeError:
            result["total"] = 0
            return result  # Return result with is_valid_json = False
        else:
            if isinstance(y_pred, str):
                result["total"] = 0
                return result  # Return result with is_valid_json = False
            result["is_valid_json"] = True

            categories_true = y_true.get("categories", {})
            categories_pred = y_pred.get("categories", {})

            if isinstance(categories_true, dict) and isinstance(categories_pred, dict):
                correct = sum(
                    categories_true.get(k, False) == categories_pred.get(k, False)
                    for k in categories_true
                )
                result["correct_categories"] = correct / len(categories_true) if categories_true else 0.0
            else:
                result["correct_categories"] = 0.0  # or raise an error if you prefer

            result["correct_sentiment"] = y_pred.get("sentiment", "") == y_true.get("sentiment", "")
            result["correct_urgency"] = y_pred.get("urgency", "") == y_true.get("urgency", "")

        # Compute overall metric score
        result["total"] = sum(
            float(result[k]) for k in ["correct_categories", "correct_sentiment", "correct_urgency"]
        ) / 3.0

        return result

    def apply(self, y_pred: Any, y_true: Any):
        return self._calculate_metrics(y_pred, y_true)

    def batch_apply(self, y_preds: List[Any], y_trues: List[Any]):
        evals = [self.apply(y_pred, y_true) for y_pred, y_true in zip(y_preds, y_trues)]
        float_keys = [k for k, v in evals[0].items() if isinstance(v, (int, float, bool))]
        return {k: sum(e[k] for e in evals) / len(evals) for k in float_keys}

metric_adapter = FacilitySupportAnalyzerMetric()

### Inference Adapter
Initialize the InferenceAdapter to choose the backend Inference. Currently, we only support BedrockInferenceAdapter.

In [ ]:
from amzn_nova_prompt_optimizer.core.inference import SageMakerInferenceAdapter

inference_adapter = SageMakerInferenceAdapter(
    endpoint_name="jezhu-micro-rai-fixed-Jan21-endpoint",
    region_name="us-west-2",
    rate_limit=10
)

2026/01/29 21:37:25 INFO amzn_nova_prompt_optimizer.core.inference.sagemaker_adapter: Initialized SageMaker adapter for endpoint: jezhu-micro-rai-fixed-Jan21-endpoint


### Evaluator

The Evaluator can use the metric_adapter, prompt_adapter, and dataset_adapter to evaluate the prompt given the `model_id` to produce an evaluation score. The Evaluator internally uses the `InferenceRunner` to first generate inference results and then evaluate the output.

#### Base Model Evaluation

In [40]:
from amzn_nova_prompt_optimizer.core.evaluation import Evaluator

evaluator = Evaluator(prompt_adapter, test_set, metric_adapter, inference_adapter)

In [8]:
original_prompt_score = evaluator.aggregate_score(model_id=NOVA_MODEL_ID)

print(f"Original Prompt Evaluation Score = {original_prompt_score}")

2026/01/29 19:33:26 INFO amzn_nova_prompt_optimizer.core.evaluation: Cache miss - Running new inference on Dataset
Running inference: 100%|██████████| 100/100 [01:30<00:00,  1.11it/s]
2026/01/29 19:34:57 INFO amzn_nova_prompt_optimizer.core.evaluation: Running Batch Evaluation on Dataset, using `batch_apply` metric
2026/01/29 19:34:57 INFO amzn_nova_prompt_optimizer.core.evaluation: Using cached inference results
2026/01/29 19:34:57 INFO amzn_nova_prompt_optimizer.core.evaluation: Running Evaluation on Dataset, using `apply` metric


Original Prompt Evaluation Score = {'is_valid_json': 1.0, 'correct_categories': 0.917, 'correct_sentiment': 0.64, 'correct_urgency': 0.66, 'total': 0.7390000000000001}


### Optimization Adapter

We can now define the Optimization Functions. The Optimization function takes as input the Prompt Adapter and Optionally a Dataset Adapter, Inference Adapter, and Metric Adapter. The optimization function optimizes the prompt and returns a Prompt Adapter.

In [9]:
class FacilitySupportAnalyzerNovaMetric(FacilitySupportAnalyzerMetric):
    def apply(self, y_pred: Any, y_true: Any):
        # Requires to return a value and not a JSON payload
        return self._calculate_metrics(y_pred, y_true)["total"]
        
    def batch_apply(self, y_preds: List[Any], y_trues: List[Any]):
        pass
nova_metric_adapter = FacilitySupportAnalyzerNovaMetric()

#### NovaPromptOptimizer

NovaPromptOptimizer = Nova Meta Prompter + MIPROv2 with Nova Model Tips

In [22]:
from amzn_nova_prompt_optimizer.core.optimizers import NovaPromptOptimizer

nova_prompt_optimizer = NovaPromptOptimizer(prompt_adapter=prompt_adapter, inference_adapter=inference_adapter, dataset_adapter=train_set, metric_adapter=nova_metric_adapter)

optimized_prompt_adapter = nova_prompt_optimizer.optimize(mode="micro")

2026/01/29 20:45:45 INFO amzn_nova_prompt_optimizer.core.optimizers.nova_prompt_optimizer.nova_prompt_optimizer: No meta_prompt_inference_adapter provided. Creating default BedrockInferenceAdapter with Nova 2.0 Lite for meta-prompting.
2026/01/29 20:45:45 INFO amzn_nova_prompt_optimizer.core.optimizers.nova_prompt_optimizer.nova_prompt_optimizer: Using separate inference adapters: Meta-prompting with BedrockInferenceAdapter, Task optimization with SageMakerInferenceAdapter
2026/01/29 20:45:45 INFO amzn_nova_prompt_optimizer.core.optimizers.nova_meta_prompter.nova_mp_optimizer: Optimizing prompt using Nova Meta Prompter with Model: us.amazon.nova-2-lite-v1:0
2026/01/29 20:45:47 INFO amzn_nova_prompt_optimizer.core.optimizers.miprov2.miprov2_optimizer: Using us.amazon.nova-micro-v1:0 for Evaluation
2026/01/29 20:45:47 INFO amzn_nova_prompt_optimizer.core.optimizers.miprov2.miprov2_optimizer: Using us.amazon.nova-2-lite-v1:0 for Prompting
2026/01/29 20:45:47 INFO amzn_nova_prompt_optimize

Bootstrapping set 1/20
Bootstrapping set 2/20
Bootstrapping set 3/20


  8%|▊         | 4/50 [00:12<02:27,  3.20s/it]


Bootstrapped 4 full traces after 4 examples for up to 1 rounds, amounting to 4 attempts.
Bootstrapping set 4/20


  8%|▊         | 4/50 [00:12<02:19,  3.04s/it]


...

2026/01/29 21:31:51 INFO dspy.evaluate.evaluate: Average Metric: 24.133333333333336 / 35 (69.0%)
2026/01/29 21:31:51 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 68.95 on minibatch of size 35 with parameters ['Predictor 0: Instruction 8', 'Predictor 0: Few-Shot Set 6'].
2026/01/29 21:31:51 INFO dspy.teleprompt.mipro_optimizer_v2: Minibatch scores so far: [73.62, 83.05, 72.86, 81.24, 75.14, 76.1, 67.71, 72.29, 81.43, 79.52, 75.9, 82.29, 73.05, 82.57, 77.81, 81.14, 76.95, 80.76, 78.67, 79.43, 72.48, 71.33, 77.33, 66.86, 82.95, 77.14, 76.0, 71.52, 68.95]
2026/01/29 21:31:51 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [71.07, 81.27, 80.8, 79.33, 76.87, 74.27]
2026/01/29 21:31:51 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 81.27
2026/01/29 21:31:51 INFO dspy.teleprompt.mipro_optimizer_v2: ==========================================


2026/01/29 21:31:51 INFO dspy.teleprompt.mipro_optimizer_v2: == Trial 36 / 37 - Minibatch ==



Average Metric: 27.33 / 35 (78.1%): 100%|██████████| 35/35 [00:59<00:00,  1.71s/it]

2026/01/29 21:32:51 INFO dspy.evaluate.evaluate: Average Metric: 27.33333333333334 / 35 (78.1%)
2026/01/29 21:32:51 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 78.1 on minibatch of size 35 with parameters ['Predictor 0: Instruction 8', 'Predictor 0: Few-Shot Set 7'].
2026/01/29 21:32:51 INFO dspy.teleprompt.mipro_optimizer_v2: Minibatch scores so far: [73.62, 83.05, 72.86, 81.24, 75.14, 76.1, 67.71, 72.29, 81.43, 79.52, 75.9, 82.29, 73.05, 82.57, 77.81, 81.14, 76.95, 80.76, 78.67, 79.43, 72.48, 71.33, 77.33, 66.86, 82.95, 77.14, 76.0, 71.52, 68.95, 78.1]
2026/01/29 21:32:51 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [71.07, 81.27, 80.8, 79.33, 76.87, 74.27]
2026/01/29 21:32:51 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 81.27
2026/01/29 21:32:51 INFO dspy.teleprompt.mipro_optimizer_v2: ==========================================


2026/01/29 21:32:51 INFO dspy.teleprompt.mipro_optimizer_v2: ===== Trial 37 / 37 - Full Evaluation =====
2026/


Average Metric: 40.03 / 50 (80.1%): 100%|██████████| 50/50 [01:22<00:00,  1.64s/it]

2026/01/29 21:34:13 INFO dspy.evaluate.evaluate: Average Metric: 40.033333333333324 / 50 (80.1%)
2026/01/29 21:34:13 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [71.07, 81.27, 80.8, 79.33, 76.87, 74.27, 80.07]
2026/01/29 21:34:13 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 81.27
2026/01/29 21:34:13 INFO dspy.teleprompt.mipro_optimizer_v2: =======================
2026/01/29 21:34:13 INFO dspy.teleprompt.mipro_optimizer_v2: 

2026/01/29 21:34:13 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 81.27!


In [31]:
optimized_prompt_adapter.show()

2026/01/29 21:36:29 INFO amzn_nova_prompt_optimizer.core.input_adapters.prompt_adapter: 
Standardized Prompt:


{
  "user_prompt": {
    "variables": [
      "input"
    ],
    "template": "{{ input }}",
    "metadata": {
      "format": "text"
    }
  },
  "system_prompt": {
    "variables": [],
    "template": "]\nTask: Analyze and categorize the incoming text to determine urgency, sentiment, and relevant categories.\n\nContext:\n- The input will be analyzed to determine urgency, sentiment, and relevant categories.\n- Urgency must be classified as one of `high`, `medium`, or `low`.\n- Sentiment must be classified as one of `negative`, `neutral`, or `positive`.\n- Categories must be a dictionary with keys from the list: `emergency_repair_services`, `routine_maintenance_requests`, `quality_and_safety_concerns`, `specialized_cleaning_services`, `general_inquiries`, `sustainability_and_environmental_practices`, `training_and_support_requests`, `cleaning_services_scheduling`, `customer_feedback_and_complaints`, `facility_management_issues`. Each category will have a boolean value indicating whether

### Optimized System Prompt

In [32]:
print(optimized_prompt_adapter.system_prompt)

]
Task: Analyze and categorize the incoming text to determine urgency, sentiment, and relevant categories.

Context:
- The input will be analyzed to determine urgency, sentiment, and relevant categories.
- Urgency must be classified as one of `high`, `medium`, or `low`.
- Sentiment must be classified as one of `negative`, `neutral`, or `positive`.
- Categories must be a dictionary with keys from the list: `emergency_repair_services`, `routine_maintenance_requests`, `quality_and_safety_concerns`, `specialized_cleaning_services`, `general_inquiries`, `sustainability_and_environmental_practices`, `training_and_support_requests`, `cleaning_services_scheduling`, `customer_feedback_and_complaints`, `facility_management_issues`. Each category will have a boolean value indicating whether it is the best matching support category tag.

Instructions:
- Carefully analyze the input to accurately determine urgency and sentiment.
- Match the input to the most appropriate category from the provided li

### Optimized User Prompt

In [33]:
print(optimized_prompt_adapter.user_prompt)

{{ input }}


### Few Shot Examples

In [34]:
print(f"Number of Few-Shot Examples = {len(optimized_prompt_adapter.few_shot_examples)}")

Number of Few-Shot Examples = 4


In [35]:
# Print only the first example
print(optimized_prompt_adapter.few_shot_examples[0])

{'input': 'Subject: Scheduled Maintenance Request for Bathroom Plumbing\n\nDear ProCare Facility Solutions Support Team,\n\nI hope this message finds you well. My name is [Sender], and I have been utilizing your services for my residential property for some time now. I have always appreciated the quality and reliability of your maintenance services.\n\nI am writing to inform you of a minor issue that has recently come up. There seems to be a small problem with the plumbing in my bathroom. While it is not an urgent matter, I would like to have it addressed at your earliest convenience to prevent any potential complications.\n\nI have not taken any steps to resolve the issue myself, as I trust your team’s expertise in handling such matters. Could you please arrange for a technician to visit my home and take a look at the problem as part of your scheduled maintenance services?\n\nThank you for your attention to this matter. I look forward to your prompt response.\n\nBest regards,\n[Sender

### Evaluator

Now we evaluate the Nova Prompt Optimizer Optimized prompt

In [41]:
from amzn_nova_prompt_optimizer.core.evaluation import Evaluator

evaluator = Evaluator(optimized_prompt_adapter, test_set, metric_adapter, inference_adapter)

In [42]:
evaluation_score = evaluator.aggregate_score(model_id=NOVA_MODEL_ID)
print(f"Nova Prompt Optimization = {evaluation_score}")

2026/01/29 21:37:38 INFO amzn_nova_prompt_optimizer.core.evaluation: Cache miss - Running new inference on Dataset
Running inference: 100%|██████████| 100/100 [01:29<00:00,  1.12it/s]
2026/01/29 21:39:07 INFO amzn_nova_prompt_optimizer.core.evaluation: Running Batch Evaluation on Dataset, using `batch_apply` metric
2026/01/29 21:39:07 INFO amzn_nova_prompt_optimizer.core.evaluation: Using cached inference results
2026/01/29 21:39:07 INFO amzn_nova_prompt_optimizer.core.evaluation: Running Evaluation on Dataset, using `apply` metric


Nova Prompt Optimization = {'is_valid_json': 1.0, 'correct_categories': 0.8990000000000002, 'correct_sentiment': 0.69, 'correct_urgency': 0.84, 'total': 0.8096666666666666}


In [43]:
optimized_prompt_adapter.save("optimized_prompt/")